In [1]:
import os
import getpass
import re
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

GROQ_API_KEY = os.environ.get("GROQ_API_KEY") or getpass.getpass("Groq API key: ")

llm = ChatOpenAI(
    model = "openai/gpt-oss-20b",
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1"
)

In [2]:
from pathlib import Path
from pypdf import PdfReader
from docx import Document
import pandas as pd

FOLDER_PATH = Path("material")

def read_file(file_path):
    extension = file_path.suffix.lower()

    if extension == ".pdf":
        reader = PdfReader(file_path)
        return "\n".join(
            page.extract_text() or ""
            for page in reader.pages
        )

    elif extension == ".txt":
        return file_path.read_text(encoding="utf-8")

    elif extension == ".docx":
        document = Document(file_path)
        return "\n".join(
            paragraph.text
            for paragraph in document.paragraphs
        )

    elif extension == ".csv":
        return pd.read_csv(file_path).to_string(index=False)

    else:
        raise ValueError("File type not supported")


files = [
    file for file in FOLDER_PATH.iterdir()
    if file.is_file()
    and file.suffix.lower() in {".pdf", ".txt", ".docx", ".csv"}
]

if not files:
    print("No supported files found in the documents folder.")
    raise SystemExit

print("\nAvailable files:\n")

for index, file in enumerate(files, start=1):
    print(f"{index}. {file.name}")

try:
    choice = int(input("\nChoose a file number: "))

    if not 1 <= choice <= len(files):
        raise ValueError("Invalid file number")

    selected_file = files[choice - 1]

    text = read_file(selected_file)

    if not text.strip():
        raise ValueError(
            "No text extracted. The PDF might be scanned."
        )

    print(f"\nSelected: {selected_file.name}")
    print(f"Extracted {len(text)} characters.")

    
except (ValueError, IndexError) as error:
    print(f"Error: {error}")


Available files:

1. Automata 1.pdf.pdf
2. Automata 2.pdf.pdf
3. Trend AI 1.pdf.pdf
4. Trend AI 2.pdf.pdf

Selected: Automata 2.pdf.pdf
Extracted 4198 characters.


In [3]:
from dataclasses import dataclass , field
from typing import TypedDict

@dataclass
class QuizState(TypedDict):
    document: str
    topic: str

    questions: list
    current_question: int

    user_answer: str
    feedback: str

    score: int
    correct_answers: int
    wrong_answers: int

    weak_topics: list
    finished: bool

In [4]:
from pydantic import BaseModel, Field
from typing import Literal

@dataclass
class Question(BaseModel):
    topic: str

    question: str

    options: dict[str, str] = Field(
        description="Exactly four options: A, B, C, D"
    )

    correct_answer: Literal["A", "B", "C", "D"]

    explanation: str

In [13]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
import json


def create_quiz_from_document(state: QuizState):
    document = state["document"][:12000]   # قص عشان حد التوكنز في Groq

    # الاجينت يطلع الـ topic من الـ PDF لو متبعتش
    topic = (state.get("topic") or "").strip()
    if not topic:
        topic_chain = ChatPromptTemplate.from_template(
            "Read the document below and reply with ONLY its main topic "
            "as a short title (max 6 words), no punctuation, no explanation.\n\n"
            "Document:\n{document}"
        ) | llm | StrOutputParser()
        topic = topic_chain.invoke({"document": document[:4000]}).strip()

    prompt = ChatPromptTemplate.from_template(
       """
            You are an educational assessment designer.

            Create multiple-choice questions using the
            document below.

            Topic: {topic}

            Document:
            {document}

            Requirements:
            - Each question must have exactly 4 options: A, B, C, D.
            - Only one option can be correct.
            - Cover important concepts from the document.
            - Include the correct answer and a clear explanation.
            - Do not invent facts that are unsupported by the document.
            - Make questions test understanding, not just memorization.

            Return valid JSON with this structure:
            {{
                "questions": [
                    {{
                        "topic": "concept name",
                        "question": "question text",
                        "options": {{
                            "A": "option A",
                            "B": "option B",
                            "C": "option C",
                            "D": "option D"
                        }},
                        "correct_answer": "A",
                        "explanation": "why this answer is correct"
                    }}
                ]
            }}
        """
    )
    
    
    json_llm = llm.bind(response_format={"type": "json_object"})
    chain = prompt | json_llm

    data = None
    last_error = None
    for attempt in range(3):               # يحاول 3 مرات لو الـ JSON طلع مكسور
        response = chain.invoke({"document": document, "topic": topic})
        raw = response.content
        match = re.search(r"\{.*\}", raw, re.DOTALL)
        try:
            data = json.loads(match.group(0), strict=False)
            if len(data["questions"]) > 0:
                break
        except (json.JSONDecodeError, AttributeError, KeyError) as error:
            last_error = error
            print(f"Attempt {attempt + 1} failed ({error}), retrying...")
            data = None

    if data is None:
        print("Last raw output:\n", raw)
        raise ValueError(f"Model did not return valid JSON: {last_error}")



    questions = data['questions']

    return {
        "topic": topic,
        "questions": questions,            
        "current_question": 0,
        "score": 0,
        "correct_answers": 0,
        "wrong_answers": 0,
        "weak_topics": [],
        "finished": False
    }

In [14]:
def evaluate_user_answer(state : QuizState):
    
    question = state['questions'][state['current_question']]
    user_answer = state['user_answer'].strip().upper()
    correct_answer = question['correct_answer'].strip().upper()
    correct_option = question['options'][correct_answer]
    
    if user_answer not in {"A","B","C","D"} :
        state['feedback'] = "Invalid answer. Please choose A, B, C, or D."
        return state
    
    if user_answer == correct_answer:
        feedback = ("Correct!" + question['explanation'])
        return {
            'feedback' : feedback,
            'score': state['score'] + 1,
            'correct_answers': state['correct_answers'] + 1,
        }
    else:
        feedback= (
            f"""
            Incorrect. The correct answer is 
            {correct_answer}: {correct_option}.
            {question['explanation']}
            """
        )
        
        return {
            'feedback' : feedback,
            "wrong_answers" : state['wrong_answers'] + 1,
            'weak_topics' : state['weak_topics'] + [question['topic']]
        }

In [15]:
def next_question(state : QuizState):
    next_index = state['current_question'] + 1
    
    if next_index >= len(state['questions']) :
        return {
            'finished' : True
        }
        
    return {
        'current_question' : next_index,
        'feedback': "",
        'user_answer': ""
    }

In [16]:
from langgraph.types import interrupt

def ask_question(state:QuizState):
    question = state['questions'][state['current_question']]
    
    answer = interrupt({
        'question_number' : state['current_question']+1 ,
        'question' : question['question'],
        'options' : question['options']
    })
    
    return { 'user_answer' : answer}

In [17]:
def route_after_next(state : QuizState):
    if state['finished']:
        return 'final_assessment'
    
    return 'ask_question'

In [18]:
def final_assessment(state : QuizState):
    total = len(state['questions'])
    score = state['score']
    percentage = round((score / total) * 100)
    
    return {
        'feedback' : (
            f"Quiz completed. Score: {score}/{total} "
            f"({percentage}%). "
            f"Review these topics: "
            f"{', '.join(set(state['weak_topics'])) or 'None'}"
        )
    }

In [19]:
builder = StateGraph(QuizState)

builder.add_node("generate_quiz", create_quiz_from_document)
builder.add_node("ask_question", ask_question)
builder.add_node("evaluate_answer", evaluate_user_answer)
builder.add_node("next_question", next_question)
builder.add_node("final_assessment", final_assessment)

builder.add_edge(START, "generate_quiz")
builder.add_edge("generate_quiz", "ask_question")
builder.add_edge("ask_question", "evaluate_answer")
builder.add_edge("evaluate_answer", "next_question")

builder.add_conditional_edges(
    "next_question",
    route_after_next,
    {
        "ask_question": "ask_question",
        "final_assessment": "final_assessment"
    }
)

builder.add_edge("final_assessment", END)

memory = MemorySaver()

graph = builder.compile(checkpointer=memory)

In [20]:
from langgraph.types import Command

config = {"configurable": {"thread_id": "student-001"}}

result = graph.invoke(
    {
        "document": text,
        "topic": "",              
        "questions": [],
        "current_question": 0,
        "user_answer": "",
        "feedback": "",
        "score": 0,
        "correct_answers": 0,
        "wrong_answers": 0,
        "weak_topics": [],
        "finished": False
    },
    config=config
)

print("Topic:", graph.get_state(config).values["topic"])

while "__interrupt__" in result:
    q = result["__interrupt__"][0].value
    print(f"\nQ{q['question_number']}: {q['question']}")
    for key, value in q["options"].items():
        print(f"  {key}. {value}")
    answer = input("Your answer (A/B/C/D): ")
    result = graph.invoke(Command(resume=answer), config=config)

print("\n", result["feedback"])

OpenAIInvalidRequestError: Error code: 400 - {'error': {'message': "Failed to validate JSON. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'json_validate_failed', 'failed_generation': ''}}